In [1]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pandas as pd
import numpy as np
import seaborn as sns
import gseapy as gp
from gseapy import gseaplot

In [2]:
import scanpy as sc

adata = sc.read_h5ad(r"D:\Dom\Psoriasis project\4th year data\Second Round Data\Xenium outputs - second round\Resegmented Xenium Outputs\ResolVI_subclusters.h5ad")
#only considering dorsal epi/dermis
adata = adata[adata.obs['compartment'].str.contains('orsal')]
adata.obs.compartment.unique()

['dorsal_dermis', 'dorsal_epidermis']
Categories (2, object): ['dorsal_dermis', 'dorsal_epidermis']

In [3]:
from pathlib import Path
results_path = Path(r"C:\Users\dbuxton\OneDrive\Desktop\biochem\Year 4\Thesis\results figures or slides\DESEQ_results")

In [4]:
import decoupler as dc

#making pseudobulk objects for the epidermis
adata_epi = adata[adata.obs['compartment'].str.contains('_epidermis')].copy()

##Filtering genes not expressed by at least 1% of cells
min_cells = int(0.01 * adata_epi.n_obs)
sc.pp.filter_genes(adata_epi, min_cells=min_cells)


big_bulk_epidermis = dc.pp.pseudobulk(
    adata_epi,
    sample_col = 'batch_key', #defines replicates
    groups_col = None, #ignore if using batch_key, add in if using celltypes
    layer = None,
    mode = 'sum',
    empty = True
)


#making pseudobulk objects for the dermis
adata_derm = adata[adata.obs['compartment'].str.contains('_dermis')].copy()

##Filtering genes not expressed by at least 1% of cells
min_cells = int(0.01 * adata_derm.n_obs)
sc.pp.filter_genes(adata_derm, min_cells=min_cells)

big_bulk_dermis = dc.pp.pseudobulk(
    adata_derm,
    sample_col = 'batch_key', #defines replicates
    groups_col = None, #ignore if using batch_key, add in if using celltypes
    layer = None,
    mode = 'sum',
    empty = True
)

d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
#If not separating male/female objects (considering together)

bulk_subtypes = {'big_bulk_epidermis':big_bulk_epidermis, 'big_bulk_dermis':big_bulk_dermis}


In [5]:
#Creating separate objects for males and females

def isolate_sex(adata, sex_str):
    bdata = adata[adata.obs.replicate.str.contains(sex_str)].copy()
    return bdata

bulk_objects = {'big_bulk_epidermis':big_bulk_epidermis, 'big_bulk_dermis':big_bulk_dermis}

bulk_subtypes = {}
for name, obj in bulk_objects.items():
    #bulk_subtypes[name] = obj #can remove this if you don't care about non-sex differences
    for sex in ['F', 'M']:
        bulk_subtypes[f'{name}_{sex}'] = isolate_sex(obj, sex)

for name, obj in bulk_subtypes.items():
    print(name)
    print(obj)

big_bulk_epidermis_F
AnnData object with n_obs × n_vars = 15 × 2144
    obs: 'batch_key', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'region', 'z_level', 'compartment', 'replicate', 'condition', '_scvi_batch', '_scvi_labels', 'highlight', 'sex', 'psbulk_cells', 'psbulk_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mean_log1p_expression', 'n_cells'
    layers: 'psbulk_props'
big_bulk_epidermis_M
AnnData object with n_obs × n_vars = 15 × 2144
    obs: 'batch_key', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'region', 'z_level', 'compartment', 'replicate', 'condition', '_scvi_batch', '_scvi_labels', 'highlight', 'sex', 'psbulk_cells', 'psbulk_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mean_log1p_expression', 'n_cells'
    layers: 'psbulk_props'
big_bulk_dermis_F
AnnData object with n_obs × n_vars = 15 × 937
    obs: 'batch_key', 'control_p

In [6]:
mechano_genes = pd.read_csv(r"C:\Users\dbuxton\Downloads\Mechanobiology_panel_combined_with_mouse.csv", index_col = 2)
mechano_genes.index

Index(['Cdh1', 'Cdh2', 'Cdh3', 'Cdh11', 'Cldn1', 'Cldn4', 'Cldn7', 'Ctnna1',
       'Ctnnb1', 'Ctnnd1',
       ...
       'Hspg2', 'Mmp14', 'Nrep', 'Plau', 'Thy1', 'Smad2', 'Smad3', 'Smad4',
       'Tgfbr1', 'Tgfbr2'],
      dtype='object', name='mouse gene', length=283)

In [7]:
mechano_genes.index.to_list()
list = [x.lower().capitalize() for x in mechano_genes.index.to_list()]

In [8]:
gene_panel = pd.read_csv(r"C:\Users\dbuxton\Downloads\XeniumPrimeMouse5Kpan_tissue_pathways_metadata.csv", index_col = 0)
gene_panel.head(1)

,gene_id,num_codewords,num_probesets,protein_name,location,cell_type,cellchat_pathway
gene_name,,,,,,,
A1cf,ENSMUSG00000052595,2,2,APOBEC1 complementation factor (APOBEC1-stimul...,Nucleus,NaN,NaN


In [9]:
list_in5k = []

for i in list:
    if i in gene_panel.index:
        list_in5k.append(i)
    else:
        continue
list_in5k
df_in5k = pd.DataFrame(list_in5k)


In [ ]:
#use if separated by sex, otherwise use next one
import matplotlib.pyplot as plt

for name, bulk in bulk_subtypes.items():
    #make a deseq data set from your anndata
    
    ###avoid recalculating if already done###
    # if Path(results_path/ f'{name}_Ctrl_D3IMQ_DEgenes.csv').exists(): #bit jammy, but if D3 is calced, then so are D7/D10
    #     continue

    dds = DeseqDataSet(counts = bulk.X,
                   metadata = bulk.obs,
                   design_factors = 'condition')
    dds.var = bulk.var


    #run deseq2 on it
    try:
        dds.deseq2()
    except ValueError as e:
        print(f'{e}_{name} failed :(')
        continue

    #get the stats
    for day in ['D7IMQ']:
        print(name.split("_")[3])
        print("="*80)


        stat_res = DeseqStats(dds, contrast = ['condition', day, 'Ctrl']) #pairwise comparison of days with each condition
        stat_res.summary()
        #get diffexp dataframe
        res = stat_res.results_df
        res_file = f'{name}_Ctrl_{day}_DEgenes.csv'
        #res.to_csv(results_path / res_file) #save table with stats of DE genes
        
        #filtering any genes with baseMean <10, choice is somewhat arbitrary
        res = res[res.baseMean >10]
        #then filter all those that are not significant or large enough
        sigs = res[(abs(res.log2FoldChange) > 1)&(res.padj <0.05)] #again, log2fc threshold is bit arbitrary,being stringent so using 1.

        
        
        # #plot CLUSTERMAP for significant genes
        cluster_map_name = f'{name}_Ctrl_{day}_wholemechano_cluster.png'
        cluster_map_path = results_path / 'clustermaps' / cluster_map_name
        if len(sigs) >2:
            dds.layers['log1p'] = np.log1p(dds.layers['normed_counts'])
            #check for valid genes
            valid_genes = [g for g in df_in5k[0] if g in dds.var_names]

            dds_sigs = dds[:, valid_genes].copy()

            grapher = pd.DataFrame(dds_sigs.layers['log1p'].T,
                        index = dds_sigs.var_names, columns = dds_sigs.obs_names)
            
            clustermap = sns.clustermap(grapher, z_score = 0, cmap = 'RdYlBu_r')
            clustermap.ax_heatmap.tick_params(axis = 'y', labelsize = '20')
            clustermap.ax_heatmap.tick_params(axis = 'x', labelsize = '20')


            clustermap.savefig(cluster_map_path, bbox_inches = 'tight', dpi = 400)
            plt.close(clustermap.fig)
        else:
            print(f'{name} failed at clustermapping')

    
        # #GSEA 


        # ranking = res['stat'].dropna().sort_values(ascending = False)

        # pre_res = gp.prerank(rnk = ranking,
        #                      gene_sets = {'list_in5k': list_in5k},
        #                      seed = 6, permutation_num = 100)
        
        # if name.split("_")[3] == 'F':
        #     sex = 'female'
        # else:
        #     sex = 'male'

        # gseaplot(pre_res.ranking, **pre_res.results['list_in5k'])
        # break

C:\Users\dbuxton\AppData\Local\Temp\ipykernel_22852\1843694547.py:10: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts = bulk.X,
Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 0.41 seconds.

Fitting dispersion trend curve...
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\pydeseq2\dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.05 seconds.

Fitting MAP dispersions...
... done in 0.37 seconds.

Fitting LFCs...
... done in 0.32 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



IndexError: list index out of range

In [16]:
import matplotlib.pyplot as plt

for name, bulk in bulk_subtypes.items():
    #make a deseq data set from your anndata
    
    ###avoid recalculating if already done###
    # if Path(results_path/ f'{name}_Ctrl_D3IMQ_DEgenes.csv').exists(): #bit jammy, but if D3 is calced, then so are D7/D10
    #     continue

    dds = DeseqDataSet(counts = bulk.X,
                   metadata = bulk.obs,
                   design_factors = 'condition')
    dds.var = bulk.var


    #run deseq2 on it
    try:
        dds.deseq2()
    except ValueError as e:
        print(f'{e}_{name} failed :(')
        continue

    #get the stats
    for day in ['D7IMQ']:

        stat_res = DeseqStats(dds, contrast = ['condition', day, 'Ctrl']) #pairwise comparison of days with each condition
        stat_res.summary()
        #get diffexp dataframe
        res = stat_res.results_df
        res_file = f'{name}_Ctrl_{day}_DEgenes.csv'
        #res.to_csv(results_path / res_file) #save table with stats of DE genes
        
        #filtering any genes with baseMean <10, choice is somewhat arbitrary
        res = res[res.baseMean >10]
        #then filter all those that are not significant or large enough
        sigs = res[(abs(res.log2FoldChange) > 1)&(res.padj <0.05)] #again, log2fc threshold is bit arbitrary,being stringent so using 1.

        
        
        # #plot CLUSTERMAP for significant genes
        cluster_map_name = f'{name}_Ctrl_{day}_wholemechano_cluster.png'
        cluster_map_path = results_path / 'clustermaps' / cluster_map_name
        if len(sigs) >2:
            dds.layers['log1p'] = np.log1p(dds.layers['normed_counts'])
            #check for valid genes
            valid_genes = [g for g in df_in5k[0] if g in dds.var_names]

            dds_sigs = dds[:, valid_genes].copy()

            grapher = pd.DataFrame(dds_sigs.layers['log1p'].T,
                        index = dds_sigs.var_names, columns = dds_sigs.obs_names)
            
            clustermap = sns.clustermap(grapher, z_score = 0, cmap = 'RdYlBu_r')
            clustermap.ax_heatmap.tick_params(axis = 'y', labelsize = '20')
            clustermap.ax_heatmap.tick_params(axis = 'x', labelsize = '20')


            clustermap.savefig(cluster_map_path, bbox_inches = 'tight', dpi = 400)
            plt.close(clustermap.fig)
        else:
            print(f'{name} failed at clustermapping')

    
        # #GSEA 


        # ranking = res['stat'].dropna().sort_values(ascending = False)

        # pre_res = gp.prerank(rnk = ranking,
        #                      gene_sets = {'list_in5k': list_in5k},
        #                      seed = 6, permutation_num = 100)
        
        # if name.split("_")[3] == 'F':
        #     sex = 'female'
        # else:
        #     sex = 'male'

        # gseaplot(pre_res.ranking, **pre_res.results['list_in5k'])
        # break

C:\Users\dbuxton\AppData\Local\Temp\ipykernel_22852\1886304705.py:10: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts = bulk.X,
Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 0.37 seconds.

Fitting dispersion trend curve...
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\pydeseq2\dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.37 seconds.

Fitting LFCs...
... done in 0.28 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.24 seconds.



Log2 fold change & Wald test p-value: condition D7IMQ vs Ctrl
            baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Aatf       89.649090       -0.390327  0.161874 -2.411301  1.589571e-02   
Abca1      45.415647       -0.824275  0.235144 -3.505411  4.559037e-04   
Abca3      44.003672       -1.797402  0.298026 -6.031027  1.629209e-09   
Abca7     109.373211       -0.866994  0.116148 -7.464572  8.357057e-14   
Abcb8      21.522924        1.499170  0.344618  4.350230  1.359947e-05   
...              ...             ...       ...       ...           ...   
Zfyve9     34.230270       -0.045068  0.209189 -0.215442  8.294224e-01   
Zkscan1    35.886741       -0.727993  0.189778 -3.836023  1.250425e-04   
Zmpste24   29.303583        0.447488  0.274523  1.630055  1.030898e-01   
Zmynd19    22.545709       -0.436927  0.277497 -1.574532  1.153645e-01   
Zzef1      58.518522       -0.279102  0.151790 -1.838744  6.595291e-02   

                  padj  
Aatf      3.095404e-02  

C:\Users\dbuxton\AppData\Local\Temp\ipykernel_22852\1886304705.py:10: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts = bulk.X,
Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.20 seconds.

Fitting dispersion trend curve...
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\pydeseq2\dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.21 seconds.

Fitting LFCs...
... done in 0.22 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.19 seconds.



Log2 fold change & Wald test p-value: condition D7IMQ vs Ctrl
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
Aatf     24.769587       -0.621409  0.255214 -2.434859  0.014898  0.054957
Abca1    69.097989        0.142823  0.184609  0.773651  0.439137  0.636952
Abca7    32.742050       -0.096628  0.202617 -0.476896  0.633436  0.780960
Abcc1    15.353058       -0.173442  0.302050 -0.574216  0.565822  0.725274
Abcf1    17.292778       -0.591944  0.258074 -2.293697  0.021808  0.072832
...            ...             ...       ...       ...       ...       ...
Zc3h11a  20.562554       -0.159566  0.253425 -0.629640  0.528930  0.705994
Zdhhc8   16.875022       -0.521512  0.264198 -1.973942  0.048388  0.132186
Zeb2     30.637227        0.068855  0.217034  0.317256  0.751050  0.855941
Zfp106   28.295164       -0.286218  0.233588 -1.225312  0.220458  0.399553
Zzef1    16.995262        0.513974  0.291399  1.763813  0.077763  0.190539

[937 rows x 6 columns]


In [ ]:
import matplotlib.pyplot as plt

for name, bulk in bulk_subtypes.items():
    #make a deseq data set from your anndata
    
    ###avoid recalculating if already done###
    # if Path(results_path/ f'{name}_Ctrl_D3IMQ_DEgenes.csv').exists(): #bit jammy, but if D3 is calced, then so are D7/D10
    #     continue

    dds = DeseqDataSet(counts = bulk.X,
                   metadata = bulk.obs,
                   design_factors = 'condition')
    dds.var = bulk.var


    #run deseq2 on it
    try:
        dds.deseq2()
    except ValueError as e:
        print(f'{e}_{name} failed :(')
        continue

    #get the stats
    for day in ['D7IMQ']:
        print(name.split("_")[3])
        print("="*80)


        stat_res = DeseqStats(dds, contrast = ['condition', day, 'Ctrl']) #pairwise comparison of days with each condition
        stat_res.summary()
        #get diffexp dataframe
        res = stat_res.results_df
        res_file = f'{name}_Ctrl_{day}_DEgenes.csv'
        #res.to_csv(results_path / res_file) #save table with stats of DE genes
        
        #filtering any genes with baseMean <10, choice is somewhat arbitrary
        res = res[res.baseMean >10]
        #then filter all those that are not significant or large enough
        sigs = res[(abs(res.log2FoldChange) > 1)&(res.padj <0.05)] #again, log2fc threshold is bit arbitrary,being stringent so using 1.

        
        
        # #plot CLUSTERMAP for significant genes
        cluster_map_name = f'{name}_Ctrl_{day}__mechano_cluster.png'
        cluster_map_path = results_path / 'clustermaps' / cluster_map_name
        if len(sigs) >2:
            dds.layers['log1p'] = np.log1p(dds.layers['normed_counts'])
            #check for valid genes
            valid_genes = [g for g in df_in5k[0] if g in dds.var_names]

            dds_sigs = dds[:, valid_genes].copy()

            grapher = pd.DataFrame(dds_sigs.layers['log1p'].T,
                        index = dds_sigs.var_names, columns = dds_sigs.obs_names)
            
            clustermap = sns.clustermap(grapher, z_score = 0, cmap = 'RdYlBu_r')

            clustermap.ax_heatmap.tick_params(axis='y', labelsize=20) # TOGGLE SIZE HERE
    
            clustermap.ax_heatmap.tick_params(axis='x', labelsize=20)

           # clustermap.savefig(cluster_map_path, bbox_inches = 'tight', dpi = 400)
            plt.close(clustermap.fig)
        else:
            print(f'{name} failed at clustermapping')

        # #GSEA 


        # ranking = res['stat'].dropna().sort_values(ascending = False)

        # pre_res = gp.prerank(rnk = ranking,
        #                      gene_sets = {'list_in5k': list_in5k},
        #                      seed = 6, permutation_num = 100)
        
        # if name.split("_")[3] == 'F':
        #     sex = 'female'
        # else:
        #     sex = 'male'

        # gseaplot(pre_res.ranking, **pre_res.results['list_in5k'])
        # break